# Surrogate test notebook (3-channel model)
Ordered top-to-bottom. Run cells in order.
Imports the model/dataset classes from `models.py`, recovers the SAME sequential
split as `main.py` (no shuffle; rows are pre-ordered train|val|test by disjoint
geomodels), loads the trained weights from `case0/`, and evaluates on the
held-out test split (unseen geomodels).

In [ ]:
# 1. Imports & config
import os, json
import numpy as np
import h5py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

from models import GeoToRatesModel, GeoRatesDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

# ---- EDIT THESE ----
dataset_dir  = "../dataset/dataset_chunks/"
dataset_file = "paired_dataset_combined.h5"
rates_file   = "all_data_all_times.npy"
case_dir     = "case0"
weights_path = os.path.join(case_dir, "model_trained.pth")

In [ ]:
# 2. Load data + recover the SAME sequential split as main.py (NO shuffle)
with h5py.File(os.path.join(dataset_dir, dataset_file), 'r') as f:
    X = np.array(f["X"])             # (N,128,128,32,4) = [facies,poro,perm,ar]
    n_train = int(f.attrs["n_train"])
    n_val   = int(f.attrs["n_val"])
    n_test  = int(f.attrs["n_test"])
rates = np.load(os.path.join(dataset_dir, rates_file))   # (N,T,F), same row order as X

# channels-first, drop facies -> [poro, perm, ar], log-transform perm and ar
X = np.transpose(X, (0,4,1,2,3))[:, 1:].astype(np.float32)
X[:, 1] = np.log(X[:, 1])
X[:, 2] = np.log(X[:, 2])

# sequential split -- rows are pre-ordered train|val|test by disjoint geomodels
N, T, Fdim = rates.shape
assert n_train + n_val + n_test == N, "split boundaries do not sum to N"
test_idx = np.arange(n_train + n_val, N)
print("X:", X.shape, " rates:", rates.shape, " test samples (unseen geomodels):", len(test_idx))

In [ ]:
# 3. Normalize using the SAVED training stats (no refitting)
rate_stats = torch.load(os.path.join(case_dir, 'rate_scaler_stats.pt'), map_location='cpu')
mu, sd = rate_stats['mean'].numpy(), rate_stats['std'].numpy()
rates_n = torch.tensor(((rates.reshape(-1, Fdim) - mu) / sd).reshape(N, T, Fdim), dtype=torch.float32)

geo_stats = torch.load(os.path.join(case_dir, 'geo_channel_stats.pt'), map_location='cpu')
geo = torch.tensor(X, dtype=torch.float32)
geo_n = (geo - geo_stats['mean']) / geo_stats['std']

test_loader = DataLoader(
    Subset(GeoRatesDataset(geo_n, rates_n), test_idx.tolist()),
    batch_size=32, shuffle=False)

In [ ]:
# 4. Build model from saved config & load trained weights
with open(os.path.join(case_dir, "model_config.json")) as f:
    cfg = json.load(f)
print("config:", cfg)

model = GeoToRatesModel(
    conv_channels=tuple(cfg["conv_channels"]),
    latent_dim=cfg["latent_dim"],
    time_steps=T,
    lstm_hidden_size=cfg["lstm_hidden_size"],
    lstm_layers=cfg["lstm_layers"],
    bidirectional=cfg["bidirectional"],
    output_dim=Fdim,
    dropout_rate=cfg["dropout"],
    pool_output=tuple(cfg["pool_output"]),
    use_coords=cfg["use_coords"],
).to(device)

model.load_state_dict(torch.load(weights_path, map_location=device))
model.eval()
print("loaded:", weights_path)

In [ ]:
# 5. Predict on the test set
crit = nn.HuberLoss(delta=1.0)
preds_n, trues_n, test_loss = [], [], 0.0
with torch.no_grad():
    for gb, rb in test_loader:
        gb, rb = gb.to(device), rb.to(device)
        out = model(gb)
        test_loss += crit(out, rb).item() * gb.size(0)
        preds_n.append(out.cpu()); trues_n.append(rb.cpu())
test_loss /= len(test_loader.dataset)
preds_n = torch.cat(preds_n).numpy()
trues_n = torch.cat(trues_n).numpy()
print(f"test Huber loss (normalized): {test_loss:.5f}")

# de-normalize to real rate units
preds = preds_n * sd + mu
trues = trues_n * sd + mu

In [ ]:
# 6. Per-well metrics in real units
print(f"{'well':>4} {'R2':>8} {'RMSE':>10} {'MAE':>10} {'mean rate':>10}")
r2s = []
for k in range(Fdim):
    t, p = trues[:,:,k].ravel(), preds[:,:,k].ravel()
    r2 = 1 - ((t-p)**2).sum() / ((t-t.mean())**2).sum()
    r2s.append(r2)
    rmse = np.sqrt(((t-p)**2).mean()); mae = np.abs(t-p).mean()
    print(f"{k:>4} {r2:>8.4f} {rmse:>10.2f} {mae:>10.2f} {t.mean():>10.2f}")
print(f"\nmean R2: {np.mean(r2s):.4f}   min R2 (worst well): {np.min(r2s):.4f}")

In [ ]:
# 7. Best / median / worst test samples by per-sample MSE
per_sample_mse = ((preds - trues)**2).mean(axis=(1,2))
order = np.argsort(per_sample_mse)
picks = {"best": order[0], "median": order[len(order)//2], "worst": order[-1]}

for name, i in picks.items():
    fig, axes = plt.subplots(3, 5, figsize=(16, 8), sharex=True)
    for k, ax in enumerate(axes.ravel()):
        ax.plot(trues[i,:,k], color="black", label="true")
        ax.plot(preds[i,:,k], color="red",   label="pred")
        ax.set_title(f"rate {k}", fontsize=9)
    axes[0,0].legend(fontsize=8)
    fig.suptitle(f"{name} test sample (idx {i}, MSE {per_sample_mse[i]:.1f})")
    fig.tight_layout(); plt.show()

In [ ]:
# 8. Parity plot (all wells, all times) + error vs time
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.scatter(trues.ravel(), preds.ravel(), s=2, alpha=0.15)
lims = [min(trues.min(), preds.min()), max(trues.max(), preds.max())]
ax1.plot(lims, lims, 'k--', lw=1)
ax1.set_xlabel("true rate"); ax1.set_ylabel("predicted rate"); ax1.set_title("parity (test set)")

rmse_t = np.sqrt(((preds - trues)**2).mean(axis=(0,2)))
ax2.plot(rmse_t)
ax2.set_xlabel("time step"); ax2.set_ylabel("RMSE (real units)"); ax2.set_title("error vs time")
plt.tight_layout(); plt.show()